# GUE H3 classification with DeepD embeddings

This notebook demonstrates the **post-embedding classification workflow** using the **H3 task from GUE** as the example.

DeepD embeddings can be obtained two ways:

1. **Compute them on demand with the DeepD inference API** — one API call per sequence provides its token-level representation directly from the raw DNA. Section 2 demos this live on one sequence from the H3 training split.
2. **Download precomputed embeddings** — the H3 train/validation/test embeddings hosted for direct download, so the classification walkthrough never needs to repeat inference.

**Workflow**

`H3 train/validation/test splits → DeepD token embeddings (section 2: live API demo; from section 3 on: precomputed) → special-token removal → masked mean pooling → MLP classifier → validation MCC model selection → test MCC / accuracy`

The sequence/label files and embeddings are hosted on the Hugging Face dataset `biomap-research/DeepD` under `fine_tune/H3/`. The notebook downloads that subtree when the files are not already present locally.

## 1. Setup

Reusable data loading, pooling, model, training, and metric implementations are kept under `embedding_classifier/src/`. The notebook provides the scientific walkthrough and calls those tested backend modules.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

# Locate the fine_tune directory (contains embedding_classifier/src).
here = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (here, *here.parents):
    if (candidate / "embedding_classifier" / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(
        "fine_tune directory not found. Open this notebook from the DeepD repo."
    )

os.chdir(PROJECT_ROOT)
BACKEND_ROOT = PROJECT_ROOT / "embedding_classifier"
sys.path.insert(0, str(BACKEND_ROOT))

CONFIG_PATH = BACKEND_ROOT / "configs" / "H3_example.yaml"
print("Project root:", PROJECT_ROOT)
print("Config:", CONFIG_PATH)


Project root: <repo_root>/notebook/fine_tune
Config: <repo_root>/notebook/fine_tune/embedding_classifier/configs/H3_example.yaml


## 2. Verify embedding extraction online (DeepD inference API)

Before connecting any data, we first show that a **per-token embedding can be computed on the fly** for any sequence: we submit one sequence from the H3 training split to the DeepD inference API (`embedding` task) and inspect the returned representation — its shape and a slice of the values.

This matters because it means the exact representations used downstream are *not* tied to a precomputed package: any H3-like sequence can be embedded from scratch with a single API call whenever needed.

After this quick live check, the notebook switches to the faster option for the full benchmark — **downloading precomputed embeddings** (section 3) — but the rest of the workflow consumes identical 2048-dimensional token embeddings either way.

In [2]:
# ---- Pick one H3 training sequence to embed via the API -----------------
import csv
from pathlib import Path

# The H3 CSVs may live next to this notebook (notebook/fine_tune/H3/) or
# under data/H3/ (the layout that the Hugging Face download in section 3 makes).
csv_candidates = [
    Path.cwd() / "H3" / "H3_train.csv",
    Path.cwd() / "data" / "H3" / "H3_train.csv",
    PROJECT_ROOT / "H3" / "H3_train.csv",
    PROJECT_ROOT / "data" / "H3" / "H3_train.csv",
]
train_csv = next((p for p in csv_candidates if p.is_file()), None)
if train_csv is None:
    # Section 3 downloads H3/; the API demo can still run with a 500-nt placeholder.
    print(
        "H3_train.csv not found yet. Using a 500-nt placeholder for the live API demo. "
        "Run section 3 to fetch biomap-research/DeepD (fine_tune/H3/) for the full benchmark."
    )
    DEMO_ID = "api_demo"
    DEMO_SEQ = ("ATG" + "ACGT" * 124 + "T")
    DEMO_LABEL = 1
    DEMO_GENOME = "GCF_000001405.40.fasta"
else:
    with open(train_csv, newline="") as f:
        first_row = next(csv.DictReader(f))

    DEMO_ID = first_row["unique_id"]
    DEMO_SEQ = first_row["nt_seq"]
    DEMO_LABEL = first_row["label"]
    DEMO_GENOME = first_row["species"]  # NCBI assembly accession, e.g. GCF_000001405.40.fasta

    print("Train CSV:", train_csv)

print("Demo sequence id:", DEMO_ID)
print("Sequence length:", len(DEMO_SEQ), "nt")
print("Label:", DEMO_LABEL)
print("Source assembly:", DEMO_GENOME)

Train CSV: <repo_root>/notebook/fine_tune/H3/H3_train.csv
Demo sequence id: <demo_sample_id>
Sequence length: 500 nt
Label: 1
Source assembly: GCF_000001405.40.fasta


In [3]:
# ---- Resolve the source genome to a scientific name ----------------------
import json
import os
import subprocess
import sys
import time

# The H3 "species" column stores an NCBI assembly accession such as
# GCF_000001405.40.fasta (human/GRCh38, the only assembly in H3). The species
# table shipped with the apiexample client maps accessions (species_id) back to
# the scientific name that the CLI accepts.
accession = DEMO_GENOME.removesuffix(".fasta")

# Locate the repository root by walking up for apiexample/cli.py.
repo = Path.cwd().resolve()
while not (repo / "apiexample" / "cli.py").is_file():
    if repo.parent == repo:
        raise FileNotFoundError("cannot locate apiexample/cli.py")
    repo = repo.parent

with open(repo / "apiexample" / "data" / "species_name_to_id.json", encoding="utf-8") as f:
    species_doc = json.load(f)
species_name = next(
    (
        name
        for name, rec in species_doc["species"].items()
        if str(rec.get("species_id")) == accession
    ),
    None,
)
if species_name is None:
    raise KeyError(f"no species record for assembly accession {accession!r}")
print("Resolved species:", species_name, f"(assembly {accession})")

# ---- Submit ONE sequence to the DeepD inference API (embedding task) ------
# API token from INFERENCE_API_TOKEN or the git-ignored API-Key.txt.
token = os.environ.get("INFERENCE_API_TOKEN", "").strip()
if not token:
    key_file = repo / "API-Key.txt"
    if key_file.is_file():
        for line in key_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and all(ord(c) < 128 for c in line):
                token = line
                break
if not token:
    raise RuntimeError("API token not found: set INFERENCE_API_TOKEN or provide API-Key.txt")

# Download the {task_id}.json result under results/api_demo/ (git-ignored).
result_dir = Path.cwd() / "results" / "api_demo"
result_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(repo / "apiexample" / "cli.py"),
    "--prompt", DEMO_SEQ,
    "--task-type", "embedding",
    "--species", species_name,
    "--output-dir", str(result_dir),
    "--timeout", "1800",
]

# Cache the result under a stable per-sequence name so re-running this notebook
# does not resubmit the job; delete the file to force a fresh API call.
cache_file = result_dir / f"{DEMO_ID}_embedding.json"
if cache_file.is_file():
    print("Reusing cached embedding result:", cache_file)
else:
    print("Submitting one sequence to the DeepD embedding API ...")
    env = dict(os.environ, INFERENCE_API_TOKEN=token)
    t0 = time.time()
    proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=1800)
    if proc.returncode != 0:
        raise RuntimeError(proc.stdout[-2000:] + proc.stderr[-2000:])
    newest = max(result_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
    cache_file.write_bytes(newest.read_bytes())
    print(f"API job finished in {time.time() - t0:.1f}s")

# ---- Inspect the returned embedding (shape + slice) -----------------------
import numpy as np

with open(cache_file, encoding="utf-8") as f:
    doc = json.load(f)
inner = doc.get("result", doc)
while isinstance(inner, dict) and "result" in inner and isinstance(inner["result"], dict):
    inner = inner["result"]
emb = inner.get("embedding", inner)
values = emb.get("values")
shape = emb.get("shape")
if shape is None and isinstance(values, list):
    shape = [len(values), len(values[0]) if values else 0]

emb_arr = np.asarray(values, dtype=np.float32)
if emb_arr.ndim == 1:
    emb_arr = emb_arr.reshape(1, -1)
print("Result file:", cache_file)
print("Embedding shape (L tokens × D dims):", emb_arr.shape)

rows, cols = min(3, emb_arr.shape[0]), min(8, emb_arr.shape[1])
print(f"Slice of the token embedding (first {rows} positions × first {cols} dims):")
print("         " + "  ".join(f"dim{c}" for c in range(cols)))
for i in range(rows):
    nums = "  ".join(f"{emb_arr[i, c]:+.4f}" for c in range(cols))
    print(f"pos {i:<4d}  {nums}")
print("  ...")
print(f"The API returns one {emb_arr.shape[1]}-dimensional vector per token; "
      f"here {emb_arr.shape[0]} tokens for the {len(DEMO_SEQ)}-nt sequence.")
print(f"The downstream classifier consumes the same 2048-D embeddings, pooled "
      f"after trimming the configured prefix tokens.")


Resolved species: Homo sapiens (assembly GCF_000001405.40)
Submitting one sequence to the DeepD embedding API ...


API job finished in 229.7s


Result file: <repo_root>/notebook/fine_tune/results/api_demo/<demo_sample_id>_embedding.json
Embedding shape (L tokens × D dims): (501, 2048)
Slice of the token embedding (first 3 positions × first 8 dims):
         dim0  dim1  dim2  dim3  dim4  dim5  dim6  dim7
pos 0     -0.1592  +0.0289  +0.0044  +0.1426  -0.1777  +0.0598  +0.0013  +0.0192
pos 1     -0.2432  +0.1040  -0.0410  -0.7227  -0.0908  -0.1416  +0.1533  +0.7031
pos 2     -0.1416  +0.0693  +0.0172  -0.4785  -0.1592  +0.1602  -0.0879  +0.4082
  ...
The API returns one 2048-dimensional vector per token; here 501 tokens for the 500-nt sequence.
The downstream classifier consumes the same 2048-D embeddings, pooled after trimming the configured prefix tokens.


### Result: computing the embedding from scratch works

The live API call produced a `[L, 2048]` tensor — one **2048-dimensional** representation per token, read directly from the raw sequence. This is the same representation space used by the rest of this notebook (the precomputed `.pt` files store the same per-token `layernorm_embedding` tensor, minus the special-token positions trimmed in section 6).

Running one API call per sequence is perfectly feasible (this is how the precomputed embeddings were originally produced), but for a whole benchmark split — tens of thousands of sequences — it is more convenient to **download the precomputed embeddings** instead. That is the path taken from section 3 onward: identical representations, already materialised, with no need to re-run inference.

## 3. Connect the H3 data on Hugging Face

H3 splits and precomputed embeddings are downloaded from [`biomap-research/DeepD`](https://huggingface.co/datasets/biomap-research/DeepD) under the `fine_tune/` prefix. The dataset repository is reserved; contents will be uploaded in a follow-up. Until then, a local `data/H3/` directory with the same layout also works.

The expected layout is:

```text
fine_tune/
└── H3/
├── H3_train.csv
├── H3_valid.csv
├── H3_test.csv
├── H3_train_embeddings/
├── H3_valid_embeddings/
└── H3_test_embeddings/
```

If the final upload uses different embedding directory names or a different per-sample filename pattern, change only the variables in this cell.

In [ ]:
HF_REPO_ID = "biomap-research/DeepD"
HF_PREFIX = "fine_tune"
HF_SUBDIR = "H3"

TRAIN_CSV_NAME = "H3_train.csv"
VAL_CSV_NAME = "H3_valid.csv"
TEST_CSV_NAME = "H3_test.csv"

TRAIN_EMBEDDING_DIRNAME = "H3_train_embeddings"
VAL_EMBEDDING_DIRNAME = "H3_valid_embeddings"
TEST_EMBEDDING_DIRNAME = "H3_test_embeddings"

# Change this only if the uploaded per-sample embedding filenames use another pattern.
EMBEDDING_FILENAME_FMT = "embedding_{unique_id}.pt"

DATA_BASE = PROJECT_ROOT / "data"
H3_DATA_ROOT = DATA_BASE / HF_SUBDIR

print("Hugging Face dataset:", HF_REPO_ID)
print("Expected local H3 directory:", H3_DATA_ROOT)


In [ ]:
# Download only the fine_tune/H3 portion of biomap-research/DeepD.
# If data/H3 already exists locally, this step can be skipped.

if (H3_DATA_ROOT / TRAIN_CSV_NAME).is_file():
    print("Using local H3 directory:", H3_DATA_ROOT)
else:
    try:
        import shutil
        from huggingface_hub import snapshot_download

        snapshot = Path(
            snapshot_download(
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                allow_patterns=[f"{HF_PREFIX}/{HF_SUBDIR}/**"],
                token=os.getenv("HF_TOKEN") or None,
            )
        )
        source = snapshot / HF_PREFIX / HF_SUBDIR
        if not source.is_dir():
            print(
                f"{HF_REPO_ID} has no {HF_PREFIX}/{HF_SUBDIR}/ yet. "
                "The dataset will be populated in a follow-up upload. "
                "Place H3 files under data/H3/ to run the remaining cells locally."
            )
        else:
            H3_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
            for path in source.rglob("*"):
                if not path.is_file():
                    continue
                target = H3_DATA_ROOT / path.relative_to(source)
                if target.exists():
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
            print("H3 data downloaded to:", H3_DATA_ROOT)
    except Exception as exc:
        print(
            f"Could not download {HF_REPO_ID} ({exc}). "
            "The dataset will be populated in a follow-up upload. "
            "Place H3 files under data/H3/ to run the remaining cells locally."
        )


## 4. Load the H3 classification configuration

H3 is treated as a binary single-label classification task. The example uses the same post-embedding classifier settings as the existing training pipeline.

In [ ]:
from src.config import load_config

cfg = load_config(str(CONFIG_PATH))

# Resolve the scientific input paths from the Hugging Face/local H3 layout.
cfg.paths.train_csv = str(H3_DATA_ROOT / TRAIN_CSV_NAME)
cfg.paths.val_csv = str(H3_DATA_ROOT / VAL_CSV_NAME)
cfg.paths.test_csv = str(H3_DATA_ROOT / TEST_CSV_NAME)
cfg.paths.train_embedding_dir = str(H3_DATA_ROOT / TRAIN_EMBEDDING_DIRNAME)
cfg.paths.val_embedding_dir = str(H3_DATA_ROOT / VAL_EMBEDDING_DIRNAME)
cfg.paths.test_embedding_dir = str(H3_DATA_ROOT / TEST_EMBEDDING_DIRNAME)
cfg.data.embedding_filename_fmt = EMBEDDING_FILENAME_FMT

parameters = pd.DataFrame(
    [
        ["Example task", "GUE H3"],
        ["Task type", cfg.task.type],
        ["Mode", cfg.task.mode],
        ["Number of classes", cfg.task.num_classes],
        ["Embedding key", cfg.data.embedding_key],
        ["Embedding filename format", cfg.data.embedding_filename_fmt],
        ["Prefix tokens removed", cfg.data.skip_prefix_tokens],
        ["Input dimension", cfg.model.input_dim],
        ["Hidden dimensions", " → ".join(map(str, cfg.model.hidden_dims))],
        ["Loss", cfg.loss.name],
        ["Metrics", ", ".join(cfg.metrics.names)],
        ["Optimizer", cfg.train.optimizer],
        ["Learning rate", cfg.train.lr],
        ["Batch size", cfg.train.batch_size],
        ["Validation interval (steps)", cfg.train.eval_every_steps],
        ["Early-stopping metric", cfg.train.early_stopping_metric],
        ["Early-stopping patience", cfg.train.early_stopping_patience],
        ["Maximum training steps", cfg.train.max_steps],
    ],
    columns=["Parameter", "Value"],
)
parameters


## 5. Check the H3 train/validation/test inputs

Each CSV must contain at least:

- `unique_id`: identifier used to locate the corresponding embedding file;
- `label`: integer class label (`0` or `1` for H3).

The embedding directories must contain the corresponding precomputed DeepD embedding files.

In [ ]:
required_paths = {
    "H3_train.csv": Path(cfg.paths.train_csv),
    "H3_valid.csv": Path(cfg.paths.val_csv),
    "H3_test.csv": Path(cfg.paths.test_csv),
    "train embeddings": Path(cfg.paths.train_embedding_dir),
    "validation embeddings": Path(cfg.paths.val_embedding_dir),
    "test embeddings": Path(cfg.paths.test_embedding_dir),
}

availability = pd.DataFrame(
    [(name, str(path), path.exists()) for name, path in required_paths.items()],
    columns=["Input", "Path", "Available"],
)
DATA_AVAILABLE = bool(availability["Available"].all())
availability


In [ ]:
if DATA_AVAILABLE:
    for split in ("train", "val", "test"):
        csv_path = Path(getattr(cfg.paths, f"{split}_csv"))
        split_df = pd.read_csv(csv_path)

        required_columns = {cfg.data.id_column, "label"}
        missing = required_columns - set(split_df.columns)
        if missing:
            raise ValueError(f"{split} CSV is missing required columns: {sorted(missing)}")

        print(
            f"{split:>5}: n={len(split_df):,}, "
            f"labels={split_df['label'].value_counts().sort_index().to_dict()}"
        )
else:
    print(
        "H3 scientific inputs are not available yet. "
        "Upload fine_tune/H3/ to biomap-research/DeepD, or place the files under data/H3/. "
        "Data-dependent cells below will be skipped."
    )


## 6. Inspect one precomputed DeepD embedding

The production data loader:

1. loads the configured tensor from each `.pt` file;
2. removes a leading singleton batch dimension when present;
3. removes the configured prefix/suffix special tokens;
4. keeps the remaining token-level representation for padded batching.

For sequence classification, the model performs masked mean pooling over valid positions.

In [ ]:
if DATA_AVAILABLE:
    from src.config import get_embedding_dir
    from src.dataset import EmbeddingDataset

    train_ds = EmbeddingDataset(
        cfg.paths.train_csv,
        cfg,
        require_label=True,
        embedding_dir=get_embedding_dir(cfg, "train"),
    )
    sample = train_ds[0]

    print("Sample ID:", sample["sample_id"])
    print("Label:", int(sample["label"]))
    print("Embedding shape after configured token trimming:", tuple(sample["embedding"].shape))
    print("Effective sequence length:", sample["seq_len"])
else:
    print("Skipped: H3 embeddings are not available yet.")


## 7. Sequence-level pooling

For `output_mode: sequence`, token embeddings are converted to one sequence representation using a **masked mean** over valid positions:

\[
h_{\mathrm{seq}} =
\frac{\sum_i m_i h_i}{\sum_i m_i},
\]

where \(h_i\) is the 2048-dimensional DeepD embedding at position \(i\), and \(m_i\) is the valid-token mask.

Thus:

`[B, L, 2048] → masked mean pooling → [B, 2048]`.

## 8. Build the H3 classifier

The classifier is a two-hidden-layer MLP:

`2048 → 512 → 128 → 2`

ReLU is applied after each hidden linear layer.

In [ ]:
from src.model import build_model

model = build_model(cfg)
print(model)


## 9. Train and select the model

Training uses cross-entropy loss and Adam optimization. Validation is performed every 20 steps. The checkpoint with the best **validation MCC** is retained, and early stopping is triggered after 10 validation evaluations without improvement.

After training, the backend reloads the best checkpoint and evaluates the independent H3 test split.

In [ ]:
if DATA_AVAILABLE:
    from src.trainer import train

    test_metrics = train(cfg)
    pd.DataFrame([test_metrics])
else:
    print("Skipped: H3 scientific inputs are not available yet.")


## 10. Test metrics and predictions

The H3 test split is evaluated using Matthews correlation coefficient (MCC) and accuracy. When enabled, the backend also exports the test predictions.

In [ ]:
if DATA_AVAILABLE:
    metrics_path = Path(cfg.paths.output_dir) / "test_metrics.json"
    pred_path = Path(cfg.train.test_output_csv)

    print("Metrics file:", metrics_path)
    if metrics_path.exists():
        print(metrics_path.read_text())

    print("\nPrediction file:", pred_path)
    if pred_path.exists():
        display(pd.read_csv(pred_path).head(10))
else:
    print("No H3 scientific result is shown until the real embeddings are connected.")


## Adapting the example

Only the Hugging Face/data-location block near the top should normally need editing.

- Default `HF_REPO_ID` is `biomap-research/DeepD`. Change it only for a fork.
- If necessary, update the three embedding directory names.
- If necessary, update `EMBEDDING_FILENAME_FMT` to match the released `.pt` filenames.
- Keep the H3 train/validation/test split and model settings unchanged when reproducing the H3 example.

A separate synthetic smoke test remains under `embedding_classifier/tests/`. It validates software execution only and is not part of the scientific H3 result.